# Natural Language to SQL

This notebook trains a small text-to-text model to turn a question and database schema into SQL. It uses 1,000 examples, semantic search, and LoRA fine-tuning.

## 1. Install the libraries

In [ ]:
%pip install -q "datasets>=3.0,<4.0" "transformers>=4.45,<5.0" "peft>=0.13,<0.16" "accelerate>=1.0,<2.0" "sentence-transformers>=3.0,<4.0" "scikit-learn>=1.3" "streamlit>=1.30"

## 2. Imports and setup

In [ ]:
import json
import re
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import NearestNeighbors
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments
from peft import LoraConfig, TaskType, get_peft_model

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 3. Load 1,000 examples and split the data

In [ ]:
data = load_dataset("b-mc2/sql-create-context")["train"]
data = data.shuffle(seed=SEED).select(range(1000))

split = data.train_test_split(test_size=0.2, seed=SEED)
train_data = split["train"]
rest = split["test"].train_test_split(test_size=0.5, seed=SEED)
valid_data = rest["train"]
test_data = rest["test"]

print("Train:", len(train_data))
print("Validation:", len(valid_data))
print("Test:", len(test_data))
print("\nExample question:", train_data[0]["question"])
print("Example SQL:", train_data[0]["answer"])

## 4. Prepare model inputs

In [ ]:
def make_prompt(row):
    return (
        "Write an SQL query for the question using this schema. Return only SQL.\n"
        f"Schema: {row['context']}\n"
        f"Question: {row['question']}\nSQL:"
    )

train_prompts = [make_prompt(row) for row in train_data]
train_sql = [row["answer"].strip() for row in train_data]
valid_prompts = [make_prompt(row) for row in valid_data]
valid_sql = [row["answer"].strip() for row in valid_data]
test_prompts = [make_prompt(row) for row in test_data]
test_sql = [row["answer"].strip() for row in test_data]

print(train_prompts[0])
print("Target SQL:", train_sql[0])

## 5. Create semantic search embeddings

In [ ]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
search_texts = [f"Schema: {row['context']}\nQuestion: {row['question']}" for row in train_data]
search_vectors = embedder.encode(
    search_texts, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=True
)
search_index = NearestNeighbors(metric="cosine", n_neighbors=3)
search_index.fit(search_vectors)
print("Embedding shape:", search_vectors.shape)

## 6. Test retrieval

In [ ]:
def find_examples(schema, question, k=2):
    text = f"Schema: {schema}\nQuestion: {question}"
    vector = embedder.encode([text], convert_to_numpy=True, normalize_embeddings=True)
    distances, indices = search_index.kneighbors(vector, n_neighbors=min(k, len(train_data)))
    return [train_data[int(i)] for i in indices[0]]

matches = find_examples(
    "CREATE TABLE students (id INTEGER, marks INTEGER)",
    "Find students who scored more than 90."
)
for row in matches:
    print("Question:", row["question"])
    print("SQL:", row["answer"], "\n")

## 7. Load the base model

In [ ]:
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
print("Loaded:", model_name)

## 8. Tokenize the data

In [ ]:
def tokenize_data(prompts, sql):
    inputs = tokenizer(prompts, max_length=256, truncation=True)
    targets = tokenizer(text_target=sql, max_length=128, truncation=True)
    inputs["labels"] = targets["input_ids"]
    return Dataset.from_dict(inputs)

train_tokens = tokenize_data(train_prompts, train_sql)
valid_tokens = tokenize_data(valid_prompts, valid_sql)
collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=base_model, label_pad_token_id=-100)
print("Tokenized rows:", len(train_tokens), len(valid_tokens))

## 9. Add LoRA and train

In [ ]:
lora = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q", "v"]
)
model = get_peft_model(base_model, lora)
model.print_trainable_parameters()

training_args = Seq2SeqTrainingArguments(
    output_dir="./sql_checkpoints",
    learning_rate=2e-4,
    per_device_train_batch_size=4 if torch.cuda.is_available() else 2,
    per_device_eval_batch_size=4 if torch.cuda.is_available() else 2,
    num_train_epochs=1,
    logging_steps=25,
    save_strategy="no",
    eval_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokens,
    eval_dataset=valid_tokens,
    data_collator=collator,
    processing_class=tokenizer
)
train_result = trainer.train()
valid_result = trainer.evaluate()
print("Training loss:", train_result.metrics.get("train_loss"))
print("Validation loss:", valid_result.get("eval_loss"))

## 10. Compare the base model and fine-tuned model

In [ ]:
def clean_sql(text):
    return re.sub(r"\s+", " ", text.strip().lower()).rstrip(";")

def predict(model_to_use, prompt):
    model_to_use.eval()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
    inputs = {key: value.to(next(model_to_use.parameters()).device) for key, value in inputs.items()}
    with torch.no_grad():
        output = model_to_use.generate(**inputs, max_new_tokens=128)
    return tokenizer.decode(output[0], skip_special_tokens=True).strip()

def exact_match(model_to_use, prompts, answers):
    predictions = [predict(model_to_use, prompt) for prompt in prompts]
    correct = sum(clean_sql(p) == clean_sql(a) for p, a in zip(predictions, answers))
    return correct / len(answers), predictions

base_eval = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
base_accuracy, base_predictions = exact_match(base_eval, test_prompts, test_sql)
fine_accuracy, fine_predictions = exact_match(trainer.model, test_prompts, test_sql)

comparison = pd.DataFrame({
    "Model": ["Base FLAN-T5-small", "FLAN-T5-small + LoRA"],
    "Exact-match accuracy": [base_accuracy, fine_accuracy]
})
display(comparison)

## 11. Save predictions and model files

In [ ]:
predictions = pd.DataFrame({
    "question": [row["question"] for row in test_data],
    "schema": [row["context"] for row in test_data],
    "expected_sql": test_sql,
    "base_prediction": base_predictions,
    "finetuned_prediction": fine_predictions
})
predictions.to_csv("sql_test_predictions.csv", index=False)
comparison.to_csv("sql_model_comparison.csv", index=False)

trainer.model.save_pretrained("sql_lora_adapter")
tokenizer.save_pretrained("sql_lora_adapter")

results = {
    "dataset": "b-mc2/sql-create-context",
    "examples": len(data),
    "train": len(train_data),
    "validation": len(valid_data),
    "test": len(test_data),
    "base_model": model_name,
    "fine_tuning": "LoRA",
    "base_exact_match": base_accuracy,
    "finetuned_exact_match": fine_accuracy,
    "training_metrics": train_result.metrics,
    "validation_metrics": valid_result
}
with open("project_results.json", "w") as file:
    json.dump(results, file, indent=2, default=str)

print("Saved CSV results, project_results.json, and sql_lora_adapter/")

## 12. Try a new question

In [ ]:
def generate_sql(schema, question):
    examples = find_examples(schema, question, k=2)
    prompt = (
        "Write an SQL query for the question using this schema. Return only SQL.\n"
        f"Schema: {schema}\nQuestion: {question}\n"
    )
    for row in examples:
        prompt += f"Example question: {row['question']}\nExample SQL: {row['answer']}\n"
    prompt += "SQL:"
    return predict(trainer.model, prompt)

schema = "CREATE TABLE students (id INTEGER, marks INTEGER)"
question = "Find students who scored more than 90."
print(generate_sql(schema, question))

## 13. Create a simple web interface

In [ ]:
app_code = """import streamlit as st
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

st.title("Natural Language to SQL")
st.write("Enter a database schema and a question.")
schema = st.text_area("Database schema", "CREATE TABLE students (id INTEGER, marks INTEGER)")
question = st.text_input("Question", "Find students who scored more than 90.")

@st.cache_resource
def load_model():
    name = "google/flan-t5-small"
    tokenizer = AutoTokenizer.from_pretrained(name)
    base = AutoModelForSeq2SeqLM.from_pretrained(name)
    model = PeftModel.from_pretrained(base, "sql_lora_adapter")
    model.eval()
    return tokenizer, model

if st.button("Generate SQL"):
    tokenizer, model = load_model()
    prompt = f"Write an SQL query for the question using this schema. Return only SQL.\nSchema: {schema}\nQuestion: {question}\nSQL:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=128)
    st.code(tokenizer.decode(output[0], skip_special_tokens=True), language="sql")
    st.caption("Review the SQL before running it on a database.")
"""
with open("app.py", "w", encoding="utf-8") as file:
    file.write(app_code)
print("Created app.py")
print("Run it in the folder containing sql_lora_adapter with: streamlit run app.py")

## Notes

- Exact-match accuracy checks whether the generated SQL matches the expected SQL after simple case and whitespace cleanup. It is not a complete measure of SQL correctness.
- A small dataset and one training epoch may give low accuracy. Report the results actually produced by the notebook.
- The web app needs the saved `sql_lora_adapter` folder beside `app.py`.